In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", None)  # use None instead of deprecated -1
import re, os, warnings

warnings.filterwarnings("ignore")
np.random.seed(37)



In [2]:
# Load data
PATH = "../input/"
df_train = pd.read_csv(os.path.join(PATH, "train.tsv"), sep="\t")
df_test = pd.read_csv(os.path.join(PATH, "test.tsv"), sep="\t")



In [3]:
# Simple preprocessing: lowercase and optional stemming
from nltk.stem.snowball import SnowballStemmer

stemmer = SnowballStemmer("english")


def preprocess(text):
    # lowercase, split, stem, and rejoin
    tokens = [stemmer.stem(w) for w in text.lower().split()]
    return " ".join(tokens)


train_phrases = df_train["Phrase"].astype(str).apply(preprocess).tolist()
test_phrases = df_test["Phrase"].astype(str).apply(preprocess).tolist()



In [4]:
# TF‑IDF vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.75, sublinear_tf=True)
X_all = tfidf.fit_transform(train_phrases + test_phrases)
X_train_full = X_all[: len(train_phrases)]
X_test = X_all[len(train_phrases) :]

y = df_train["Sentiment"].values



In [5]:
# Validation split to check accuracy (optional but informative)
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_full, y, test_size=0.2, random_state=42, stratify=y
)

from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(C=4.0, max_iter=1000, solver="liblinear")
logreg.fit(X_tr, y_tr)
val_acc = logreg.score(X_val, y_val)
print(f"Validation accuracy: {val_acc:.4f}")



Validation accuracy: 0.6441


In [6]:
# Train on the full training data
logreg.fit(X_train_full, y)



LogisticRegression(C=4.0, max_iter=1000, solver='liblinear')

In [7]:
# Predict test set and create submission
pred_test = logreg.predict(X_test).astype(int)
submission = pd.DataFrame({"PhraseId": df_test["PhraseId"], "Sentiment": pred_test})
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission file written to {submission_path}")

Submission file written to submission.csv
